# Устойчивость к качеству снимка — замер на готовых весах

Считает таблицу «воздействие → mAP50» для уже обученной модели, **без переобучения**: около 10 минут.

**Настройки ноутбука:** Accelerator — GPU (хватит одной), Internet — On.

**Add Input, два входа:**
1. датасет MAR20 (`military aircraft recognition`);
2. веса модели: Your Work → нужный ноутбук → его Output (там лежит `*.pt`).

Веса ищутся в `/kaggle/input` автоматически. Результат — `robustness.md` и `robustness.json`
в Output этой версии.

In [ ]:
REPO_URL = "https://github.com/miss-mississippi/muxxed_defence_tech.git"
WEIGHTS = None   # None — найти первый *.pt в /kaggle/input; иначе полный путь
LIMIT, IMGSZ = 400, 800

In [ ]:
!rm -rf /kaggle/working/repo && git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q ultralytics==8.4.152

In [ ]:
import glob
from pathlib import Path

hits = sorted(Path("/kaggle/input").glob("**/Annotations/Oriented Bounding Boxes"))
if not hits:
    raise SystemExit("MAR20 не найден в /kaggle/input: Add Input → датасет MAR20")
MAR20_SRC = str(hits[0].parent.parent)

if WEIGHTS is None:
    found = sorted(Path("/kaggle/input").glob("**/*.pt"))
    if not found:
        raise SystemExit("веса не найдены в /kaggle/input: Add Input → Your Work → Output нужного ноутбука")
    WEIGHTS = str(found[0])
print("MAR20:", MAR20_SRC, "\nвеса:", WEIGHTS)

!python scripts/prepare_mar20.py --src "$MAR20_SRC" --out /kaggle/working/mar20_yolo | tail -2

In [ ]:
!python scripts/robustness.py --data /kaggle/working/mar20_yolo/mar20.yaml \
    --weights "$WEIGHTS" --split test --limit $LIMIT --imgsz $IMGSZ --device 0

In [ ]:
import shutil
from IPython.display import Markdown, display

export = Path("/kaggle/working/export")
export.mkdir(exist_ok=True)
for f in glob.glob("outputs/robustness/robustness.*"):
    shutil.copy2(f, export)
display(Markdown(Path("outputs/robustness/robustness.md").read_text()))
print(sorted(p.name for p in export.iterdir()))